In [ ]:
stack = "TEST"
# stack = "STAGING"
# stack = "PRODUCTION"

GCP_PROJECT = "cerulean-338116"
GCP_REGION = "europe-west1"


# %load_ext autoreload
# %autoreload 2

In [ ]:
import datetime
import subprocess
import httpx
import json
import boto3
import tqdm
import os
import operator
import sys
from pathlib import Path
import pandas as pd
from google.cloud import tasks_v2
from dotenv import load_dotenv

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "cerulean_cloud").is_dir()
)
sys.path.insert(0, str(repo_root))
from cerulean_cloud.cloud_function_historical_run.inventory import (  # noqa: E402
    list_sentinel1_scene_ids,
    filter_sentinel1_scene_ids,
)

from cerulean_cloud.cloud_function_historical_run.main import (  # noqa: E402
    run_historical as historical_run,
)

from cerulean_cloud.cloud_function_historical_run.local import (  # noqa: E402
    resolve_orchestrator_queue,
)

load_dotenv(".env")


(
    NAME,
    TITILER_URL,
    TITILER_API_KEY,
    API_KEY,
    INFERENCE_URL,
    DB_URL,
    ASA_QUEUE,
    ASA_URL,
    SR_URL,
    ORCHESTRATOR_URL,
) = operator.itemgetter(
    "NAME",
    "TITILER_URL",
    "TITILER_API_KEY",
    "API_KEY",
    "INFERENCE_URL",
    "DB_URL",
    "ASA_QUEUE",
    "ASA_URL",
    "SR_URL",
    "ORCHESTRATOR_URL",
)(json.loads(os.getenv("STACK_SECRETS"))[stack])

# Run historical selection and queue submission in this notebook's Python process.
# DB_URL must be reachable locally; HISTORICAL_DB_URL can override it for a proxy.
pulumi_stack = {"TEST": "test", "STAGING": "staging", "PRODUCTION": "prod20240903"}[
    stack
]
os.environ.update(
    {
        "DB_URL": os.getenv("HISTORICAL_DB_URL", DB_URL).replace(
            "postgresql+asyncpg://", "postgresql://", 1
        ),
        "GCPPROJECT": GCP_PROJECT,
        "GCPREGION": GCP_REGION,
        "ORCHESTRATOR_URL": ORCHESTRATOR_URL,
        "API_KEY": API_KEY,
    }
)
print(f"Historical-run target: {stack}")


def read_scene_ids_csv(path):
    """Accept both current and legacy TriggerOnDemand CSV exports."""
    frame = pd.read_csv(path)
    for column in ("scene_id", "s1_scene_id"):
        if column in frame.columns:
            return frame[column].dropna().astype(str).str.strip().tolist()
    raise ValueError("CSV must contain a scene_id or s1_scene_id column")

In [ ]:
# FROM LIST
scenes = [
    "S1A_S3_GRDH_1SDV_20230807T171941_20230807T172010_049772_05FC2A_B3DA",
]

print(len(scenes))

In [ ]:
# FROM CSV (accepts scene_id or legacy s1_scene_id)
scenes_csv = "scene_ids.csv"
scenes = read_scene_ids_csv(scenes_csv)
print(len(scenes))

In [ ]:
# FROM UPSTREAM DATE RANGE (inclusive UTC acquisition dates)
# Local listing creates no batch or tasks and is not limited by the function timeout.
# Raw inventory; use the common selection options below before submission.
start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 1, 2)

scenes = list_sentinel1_scene_ids(
    start_date,
    end_date,
    modes=("IW", "EW"),  # None includes all modes
    polarizations=("DV", "SV"),  # None includes all polarizations
)
pd.DataFrame({"scene_id": scenes}).to_csv("scene_ids.csv", index=False)
print(f"{len(scenes):,} upstream scenes")

In [ ]:
# COMMON SELECTION OPTIONS — run after choosing a list, CSV, or upstream date range.
# Rerun your source cell first when changing filters to restore removed scenes.
PROCESSED_CSV = None  # Optional CSV of scenes to exclude; accepts either ID column.
APPLY_RELEVANCY = False  # True applies ocean, VV, and resolution checks.
GEOMETRY_FILE = None  # Optional region GeoJSON FeatureCollection.

if PROCESSED_CSV is not None:
    processed = set(read_scene_ids_csv(PROCESSED_CSV))
    scenes = [scene for scene in scenes if scene not in processed]

geometry = None
if GEOMETRY_FILE is not None:
    with open(GEOMETRY_FILE) as src:
        geometry = json.load(src)

scenes = filter_sentinel1_scene_ids(
    scenes, geometry=geometry, apply_relevancy=APPLY_RELEVANCY
)
print(f"{len(scenes):,} selected scenes")

# ORCHESTRATOR (local historical-run)

Run setup, choose **one** scene source above, then run the common selection options.
The common options support excluding processed scenes, optional scene-relevancy,
and an optional geographic region for every source. Relevance defaults to bypass;
a supplied region still applies. Selected IDs are deduplicated before submission.

The cell below awaits historical-run's shared Python runner locally in batches of
100. Each batch checks the database for successful runs, records a HISTORICAL_RUN
trigger, and submits directly to the selected stack's historical queue. Inference
runs in the cloud. The historical-run HTTP endpoint and Pulumi CLI are not used.
No function deployment is needed for this local path.

Local setup requires a reachable PostgreSQL DB_URL from STACK_SECRETS and Google
application default credentials with permission to list queues and create Cloud Tasks. Set
HISTORICAL_DB_URL in the environment or .env to use a local database proxy URL
instead; it must point to the selected stack's database. SQLAlchemy-style
postgresql+asyncpg URLs are converted for asyncpg automatically. Run
`gcloud auth application-default login` if local Google credentials are not set up.
Check GCP_PROJECT/GCP_REGION in the first cell and rerun setup after switching stacks.

Set `DRY_RUN = True` to queue test tasks without inference, or `False` for inference.
Local submission requires an explicit boolean rather than inheriting a cloud setting.
`MANUAL_TRIGGER = True` uses run-specific downstream source-association task names.
Each batch has its own 450-second queue-submission budget; local discovery is not
limited by the function's execution timeout.

Results report queued tasks, not completed inference. If submission fails, execution
stops and batch_results retains confirmed earlier batches. A failed batch may be
partially queued; inspect the output before resubmitting. Successful scenes are
excluded by the DB check, but queued/unfinished scenes can still be submitted again.
Keep the notebook running until submission finishes. Trigger log links are empty
for local runs; progress is printed in the notebook. ASA and GeoTIFF download
utilities below retain their separate execution paths.


In [ ]:
DRY_RUN = False  # True: queue test tasks without inference; False: run inference.
MANUAL_TRIGGER = True
BATCH_SIZE = 100

if not isinstance(DRY_RUN, bool):
    raise ValueError("DRY_RUN must be True or False for local submission")
if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be positive")

# Resolve the deployed queue once, including its Pulumi-generated suffix.
os.environ["QUEUE"] = resolve_orchestrator_queue(GCP_PROJECT, GCP_REGION, pulumi_stack)
print(f"Queue: {os.environ['QUEUE']}")

batch_results = []
num_batches = (len(scenes) + BATCH_SIZE - 1) // BATCH_SIZE
for start in range(0, len(scenes), BATCH_SIZE):
    payload = {
        "scene_ids": scenes[start : start + BATCH_SIZE],
        "manual_trigger": MANUAL_TRIGGER,
        "dry_run": DRY_RUN,
    }
    result = await historical_run(payload)
    batch_results.append(result)
    print(f"Batch {len(batch_results)}/{num_batches}: {result}")

# ASA --SLOW--

In [ ]:
import concurrent.futures
import uuid


ASA_RUN_NAME = uuid.uuid4().hex


def create_task(scene_id, parent, run_name=ASA_RUN_NAME):
    # Define the payload for the task
    payload = {
        "scene_id": scene_id,
        "dry_run": False,
        "run_name": run_name,
        "run_flags": [],  # Empty list will backfill GSP, comment out to run all
    }

    # Define the task with an HTTP request
    task_id = f"asa-{scene_id}-{run_name}"
    task = {
        "name": client.task_path("cerulean-338116", "europe-west1", ASA_QUEUE, task_id),
        "http_request": {  # Specify the type of request
            "http_method": tasks_v2.HttpMethod.POST,
            "url": ASA_URL,  # The URL to send the request to
            "headers": {
                "Content-type": "application/json",
                "Authorization": f"Bearer {API_KEY}",
            },
            "body": json.dumps(
                payload
            ).encode(),  # Convert payload to JSON and encode to bytes
        },
    }

    try:
        # Create the task in the specified queue
        response = client.create_task(request={"parent": parent, "task": task})
        response
    except Exception as e:
        print(f"Error creating task for scene_id {scene_id}: {e}")


os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)
client = tasks_v2.CloudTasksClient()
parent = client.queue_path("cerulean-338116", "europe-west1", ASA_QUEUE)

max_workers = 100  # Example of aggressive concurrency
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(
        tqdm.tqdm(
            executor.map(lambda scene_id: create_task(scene_id, parent), scenes),
            total=len(scenes),
        )
    )

# for scene_id in tqdm.tqdm(scenes):
#     create_task(scene_id, parent)

# ASA --FAST--

In [ ]:
import uuid

ASA_RUN_NAME = uuid.uuid4().hex
for scene_id in tqdm.tqdm(scenes):
    try:
        asa_result = httpx.post(
            ASA_URL,
            json={
                "scene_id": f"{scene_id}",
                "dry_run": False,
                "run_name": ASA_RUN_NAME,
                # "run_flags": ["INFRA"],
            },
            timeout=0.1,
            headers={"Authorization": f"Bearer {API_KEY}"},
        )
    except Exception:
        # print(e)
        pass

In [ ]:
[f"https://cerulean-ui-test-5qkjkyomta-uk.a.run.app/?scene_id={a}" for a in scenes]

https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Catalog/Examples.html

# AWS OpenDataRegistry ls and productInfo.json

## Download GeoTiffs from AWS

In [ ]:
# Download GeoTiffs directly from S3 Open Data Registry using request payer and a list of Scene IDs

import tqdm

s3_client = boto3.client("s3")
bucket_name = "sentinel-s1-l1c"

for scene_id in tqdm.tqdm(scenes):
    measurement = f"GRD/{scene_id[17:21]}/{int(scene_id[21:23])}/{int(scene_id[23:25])}/IW/DV/{scene_id}/measurement/"
    try:
        obj = s3_client.get_object(
            Bucket=bucket_name, Key=measurement + "iw-vv.tiff", RequestPayer="requester"
        )
        file_path = f"/Users/jonathanraphael/git/ceruleanserver/local/temp/outputs/_rasters/{scene_id}.tiff"
        with open(file_path, "wb") as file:
            chunks = []
            while data := obj["Body"].read(1024):
                chunks.append(data)
            file.write(b"".join(chunks))

        print(f"{scene_id} downloaded successfully!")
    except Exception:
        command = (
            f"aws s3 ls s3://{bucket_name}/{measurement} --request-payer requester"
        )
        output = subprocess.check_output(command, shell=True)
        print(output)
        print(f"{scene_id} is missing the VV file. Perhaps one of the above instead?")


print("wrapped")